# 双动量调仓助手（基于当前持仓给出调仓建议）

这份 notebook 复用已有的：

- `market_data.py`
- `dual_momentum.py`
- `dual_momentum_backtest.py`

目标是：

1. 定义双动量资产池  
2. 输入当前持仓（按**手**）、成本价、可用现金  
3. 按当前双动量参数，自动反推出所需历史数据读取范围  
4. 从本地数据库读取最新市场数据  
5. 计算**最新双动量目标权重 / 现金权重 / 最新 snapshot 明细**  
6. 在**整手、long-only、考虑交易成本、考虑成交额占比限制**下，给出调仓建议

## 说明

- 这里的建议基于**数据库中最新可用交易日**的数据
- `signal_date` 使用数据库里最新可用的收盘日期
- 成交价使用你选择的 `execution_price_type`（默认 `avg`）做估算
- 若某资产当日无可交易价格，则不会对其给出成交建议
- 若双动量目标未满配，剩余部分会保留为现金
- 若当前持仓里包含不在候选池/防御池中的标的，会自动纳入估值与减仓建议范围

In [ ]:
from pathlib import Path
import os
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_data import create_manager, today_str, load_tushare_token

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

# ========= 让当前目录可导入本地 py 文件 =========
cwd = str(Path.cwd().resolve())
if cwd not in sys.path:
    sys.path.insert(0, cwd)

# ========= 导入双动量回测模块 =========
dm_bt = importlib.import_module("dual_momentum_backtest")

build_market_matrices = dm_bt.build_market_matrices
get_execution_price_matrix = dm_bt.get_execution_price_matrix
compute_target_weights_on_date = dm_bt.compute_target_weights_on_date
rebalance_to_target_weights = dm_bt.rebalance_to_target_weights
calc_actual_weights = dm_bt.calc_actual_weights
calc_portfolio_value = dm_bt.calc_portfolio_value

print("dual momentum backtest module =", dm_bt.__name__)
print("module file =", dm_bt.__file__)

# ========= 连接数据库 =========
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", Path(DB_PATH).resolve())

In [ ]:
# ========= 输入区：资产池 / 当前持仓 / 成本价 / 现金 / 参数 =========

# ===== 双动量资产池 =====
CANDIDATE_ASSETS = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    "511090.SH",  # 30年国债ETF (进攻池中的利率趋势资产)
    "159981.SZ",  # 能源ETF
    "159985.SZ",  # 豆粕ETF
    "501018.SH",  # 南方原油LOF
    "515100.SH",  # 红利低波100ETF
]

DEFENSIVE_ASSETS = [
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF
]

# 当前持仓：按“手”填写，1 手 = 100 份（默认）
CURRENT_LOTS = {
    # "510300.SH": 5,
    # "511090.SH": 2,
    # "518880.SH": 1,
}

# 成本价：按“每份/每股价格”填写；没有就写 0 或删掉
COST_PRICE = {
    # "510300.SH": 4.60,
    # "511090.SH": 108.20,
}

AVAILABLE_CASH = 100000.0

# ===== 调仓参数（与回测保持一致） =====
LOT_SIZE = 100
EXECUTION_PRICE_TYPE = "avg"     # open / close / high / low / avg
FEE_RATE_BUY = 0.0005
FEE_RATE_SELL = 0.0005

# Tushare fund_daily 的 amount 单位通常是“千元”
MAX_TRADE_AMOUNT_RATIO = 0.05
AMOUNT_UNIT_SCALE = 1000.0
VALUATION_FFILL_LIMIT = 5

# ===== 双动量预处理参数 =====
DM_PREPARE_KWARGS = {
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.0,   # 调仓助手更适合兼容晚成立资产
    "drop_all_na_dates": True,
}

# ===== 双动量信号参数（与回测保持一致） =====
DM_SNAPSHOT_KWARGS = {
    "candidate_assets": CANDIDATE_ASSETS,
    "defensive_assets": DEFENSIVE_ASSETS,

    # --- 绝对动量 ---
    "abs_lookback": 60,
    "abs_threshold": 0.0,
    "abs_skip_recent": 0,
    "abs_return_type": "simple",
    "abs_ma_window": 120,
    "abs_ma_type": "sma",

    # --- 相对动量 ---
    "rel_lookbacks": [20, 60, 120],
    "rel_weights": [0.2, 0.3, 0.5],
    "rel_skip_recent": 0,
    "rel_return_type": "simple",
    "rel_risk_adjusted": False,
    "rel_vol_lookback": 20,

    # --- 组合构建 ---
    "top_k": 3,
    "weighting": "equal",      # equal / score / rank / inv_vol
    "fill_unallocated_to_defensive": True,
    "min_history": 130,
    "max_single_weight": 0.6,

    # --- 市场总开关 ---
    "market_asset": "510300.SH",
    "market_lookback": 120,
    "market_threshold": 0.0,
    "market_skip_recent": 0,
    "market_ma_window": None,
    "market_ma_type": "sma",

    # --- 其他 ---
    "use_talib": False,
}

# ===== 数据读取补充参数 =====
CHECK_EXCHANGE = "SSE"
EXTRA_WARMUP_DAYS = 40

# 会自动把当前持仓里的标的也纳入估值与调仓建议范围
WATCHLIST = list(dict.fromkeys(CANDIDATE_ASSETS + DEFENSIVE_ASSETS + list(CURRENT_LOTS.keys())))

print("candidate assets =", CANDIDATE_ASSETS)
print("defensive assets =", DEFENSIVE_ASSETS)
print("watchlist =", WATCHLIST)
print("current lots =", CURRENT_LOTS)
print("available cash =", AVAILABLE_CASH)

In [ ]:
# 1. 刷新最近 180 天数据
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=WATCHLIST,
    lookback_days=180,
    end_date=today_str(),
)

print("最近数据刷新结果：")
print(recent_summary)

In [ ]:
# ========= 按当前双动量参数反推历史读取范围，并读取本地市场数据 =========

def _to_ts(x):
    return pd.Timestamp(str(x))

def _yyyymmdd(ts):
    return pd.Timestamp(ts).strftime("%Y%m%d")

def _infer_required_trade_days(dm_kw: dict) -> int:
    need = []

    abs_lookback = int(dm_kw.get("abs_lookback", 0) or 0)
    abs_skip_recent = int(dm_kw.get("abs_skip_recent", 0) or 0)
    if abs_lookback > 0:
        need.append(abs_lookback + abs_skip_recent + 1)

    rel_lookbacks = dm_kw.get("rel_lookbacks")
    rel_lookback = dm_kw.get("rel_lookback")
    rel_skip_recent = int(dm_kw.get("rel_skip_recent", 0) or 0)
    if rel_lookbacks is not None:
        need.extend([int(x) + rel_skip_recent + 1 for x in rel_lookbacks])
    elif rel_lookback is not None:
        need.append(int(rel_lookback) + rel_skip_recent + 1)

    abs_ma_window = dm_kw.get("abs_ma_window")
    if abs_ma_window is not None:
        need.append(int(abs_ma_window))

    if bool(dm_kw.get("rel_risk_adjusted", False)):
        rel_vol_lookback = int(dm_kw.get("rel_vol_lookback", 20) or 20)
        need.append(rel_vol_lookback + 1)

    market_lookback = dm_kw.get("market_lookback")
    market_skip_recent = int(dm_kw.get("market_skip_recent", 0) or 0)
    if market_lookback is not None:
        need.append(int(market_lookback) + market_skip_recent + 1)

    market_ma_window = dm_kw.get("market_ma_window")
    if market_ma_window is not None:
        need.append(int(market_ma_window))

    min_history = dm_kw.get("min_history")
    if min_history is not None:
        need.append(int(min_history))

    return int(max(need) if len(need) > 0 else 0)

def _get_open_calendar(exchange: str, start_date: str, end_date: str) -> pd.DatetimeIndex:
    try:
        open_dates = manager.store.get_open_trade_dates(
            exchange=exchange,
            start_date=start_date,
            end_date=end_date,
        )
    except Exception:
        open_dates = None

    if open_dates is None or len(open_dates) == 0:
        return pd.DatetimeIndex([])

    cal = pd.to_datetime(pd.Series(open_dates).astype(str), format="%Y%m%d", errors="coerce")
    cal = cal.dropna().drop_duplicates().sort_values()
    return pd.DatetimeIndex(cal)

def _infer_history_start_date(end_date: str, dm_kw: dict, exchange: str = "SSE", extra_trade_days: int = 40) -> pd.Timestamp:
    end_ts = _to_ts(end_date)
    required_trade_days = _infer_required_trade_days(dm_kw)

    approx_back_days = max(180, int((required_trade_days + extra_trade_days) * 2.6))
    cal_start = _yyyymmdd(end_ts - pd.Timedelta(days=approx_back_days))
    cal_end = _yyyymmdd(end_ts)

    cal = _get_open_calendar(exchange=exchange, start_date=cal_start, end_date=cal_end)
    cal = cal[cal <= end_ts]

    if len(cal) == 0:
        # 交易日历不可用时，用一个保守的自然日回退
        return end_ts - pd.Timedelta(days=approx_back_days)

    need_back = required_trade_days + extra_trade_days
    idx = max(0, len(cal) - 1 - need_back)
    return pd.Timestamp(cal[idx])

END_DATE = today_str()
HISTORY_START_DATE = _infer_history_start_date(
    end_date=END_DATE,
    dm_kw=DM_SNAPSHOT_KWARGS,
    exchange=CHECK_EXCHANGE,
    extra_trade_days=EXTRA_WARMUP_DAYS,
)

print("end date =", END_DATE)
print("history start date =", HISTORY_START_DATE.strftime("%Y-%m-%d"))
print("required trade days =", _infer_required_trade_days(DM_SNAPSHOT_KWARGS))

raw_prices = manager.store.get_daily_prices(
    ts_codes=WATCHLIST,
    start_date=_yyyymmdd(HISTORY_START_DATE),
    end_date=END_DATE,
)

print("raw_prices shape =", raw_prices.shape)
display(raw_prices.head())

coverage = raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"])
display(coverage)

market = build_market_matrices(
    data=raw_prices,
    fields=("open", "high", "low", "close", "amount"),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

DM_PREPARE_KWARGS["calendar"] = market["close"].index

close_px = market["close"].copy()
val_px = close_px.ffill(limit=VALUATION_FFILL_LIMIT)
exec_px = get_execution_price_matrix(market, execution_price_type=EXECUTION_PRICE_TYPE)
amount_px = market["amount"].copy()

latest_signal_date = close_px.index.max()
val_today = val_px.loc[latest_signal_date]
exec_today = exec_px.loc[latest_signal_date]
amount_today = amount_px.loc[latest_signal_date]

print("latest signal date =", latest_signal_date)
print("latest available assets =", close_px.columns.tolist())

In [ ]:
# ========= 计算当前组合状态 =========
codes = close_px.columns.tolist()

ignored_holdings = [code for code in CURRENT_LOTS if code not in codes]
if len(ignored_holdings) > 0:
    print("以下当前持仓不在已读取市场数据中，已忽略：", ignored_holdings)

current_shares = pd.Series(0, index=codes, dtype=int)
for code, lots in CURRENT_LOTS.items():
    if code in current_shares.index:
        current_shares.loc[code] = int(lots) * LOT_SIZE

cost_price_series = pd.Series(0.0, index=codes, dtype=float)
for code, px in COST_PRICE.items():
    if code in cost_price_series.index:
        cost_price_series.loc[code] = float(px)

current_market_value = (current_shares * val_today.reindex(codes)).fillna(0.0)
current_actual_weights = calc_actual_weights(current_shares, val_today, cash=AVAILABLE_CASH).reindex(codes).fillna(0.0)
portfolio_value = calc_portfolio_value(current_shares, val_today, AVAILABLE_CASH)
cash_weight = AVAILABLE_CASH / portfolio_value if portfolio_value > 0 else np.nan

current_summary = pd.DataFrame({
    "lots": current_shares / LOT_SIZE,
    "shares": current_shares,
    "cost_price": cost_price_series,
    "valuation_price": val_today.reindex(codes),
    "execution_price": exec_today.reindex(codes),
    "market_value": current_market_value,
    "actual_weight": current_actual_weights,
})

current_summary["pnl"] = (current_summary["valuation_price"] - current_summary["cost_price"]) * current_summary["shares"]
current_summary["pnl_pct"] = np.where(
    current_summary["cost_price"] > 0,
    current_summary["valuation_price"] / current_summary["cost_price"] - 1.0,
    np.nan,
)

print("current portfolio value =", round(portfolio_value, 2))
print("cash =", round(AVAILABLE_CASH, 2))
print("cash weight =", round(cash_weight, 4))

display(current_summary)

In [ ]:
# ========= 计算最新双动量目标权重 / snapshot 明细 =========
snapshot = compute_target_weights_on_date(
    close_price_df=close_px,
    signal_date=latest_signal_date,
    dm_prepare_kwargs=DM_PREPARE_KWARGS,
    dm_snapshot_kwargs=DM_SNAPSHOT_KWARGS,
)

signal_valid = bool(snapshot.get("signal_valid", False))
selected_assets = snapshot.get("selected_assets", [])
defensive_assets_used = snapshot.get("defensive_assets_used", [])
market_regime_on = snapshot.get("market_regime_on", False)
target_cash_weight = float(snapshot.get("cash_weight", 0.0))
detail = snapshot.get("detail", pd.DataFrame())

target_weights = snapshot.get("target_weights", pd.Series(dtype=float)).reindex(codes).fillna(0.0)

print("signal_valid =", signal_valid)
print("market_regime_on =", market_regime_on)
print("selected_assets =", selected_assets)
print("defensive_assets_used =", defensive_assets_used)
print("target_cash_weight =", round(target_cash_weight, 4))

target_weight_df = target_weights.to_frame("target_weight")
target_weight_df.loc["CASH", "target_weight"] = target_cash_weight
display(target_weight_df.sort_values("target_weight", ascending=False))

if len(detail) > 0:
    print("snapshot 明细：")
    display(detail)

if not signal_valid:
    raise RuntimeError("当前最新日期未生成有效双动量信号，请先检查数据覆盖、参数、市场总开关或 min_history 设置。")

In [ ]:
# ========= 生成调仓建议 =========
new_shares, new_cash, trades_df, after_weights, after_cash_weight = rebalance_to_target_weights(
    current_shares=current_shares,
    cash=AVAILABLE_CASH,
    target_weights=target_weights,
    exec_prices=exec_today,
    val_prices=val_today,
    amount_series=amount_today,
    fee_rate_buy=FEE_RATE_BUY,
    fee_rate_sell=FEE_RATE_SELL,
    lot_size=LOT_SIZE,
    max_trade_amount_ratio=MAX_TRADE_AMOUNT_RATIO,
    amount_unit_scale=AMOUNT_UNIT_SCALE,
    trade_date=latest_signal_date,
)

after_market_value = (new_shares * val_today.reindex(codes)).fillna(0.0)
after_weights = after_weights.reindex(codes).fillna(0.0)

if len(trades_df) > 0:
    buy_df = trades_df[trades_df["side"] == "BUY"].groupby("ts_code")["shares"].sum().rename("buy_shares")
    sell_df = trades_df[trades_df["side"] == "SELL"].groupby("ts_code")["shares"].sum().rename("sell_shares")
    trade_value_df = trades_df.groupby("ts_code")["trade_value"].sum().rename("trade_value")
    trade_cost_df = trades_df.groupby("ts_code")["cost"].sum().rename("trade_cost")
else:
    buy_df = pd.Series(dtype=float, name="buy_shares")
    sell_df = pd.Series(dtype=float, name="sell_shares")
    trade_value_df = pd.Series(dtype=float, name="trade_value")
    trade_cost_df = pd.Series(dtype=float, name="trade_cost")

rebalance_table = pd.DataFrame(index=codes)
rebalance_table["current_lots"] = current_shares / LOT_SIZE
rebalance_table["current_shares"] = current_shares
rebalance_table["target_weight"] = target_weights
rebalance_table["current_weight"] = current_actual_weights
rebalance_table["after_weight"] = after_weights
rebalance_table["valuation_price"] = val_today.reindex(codes)
rebalance_table["execution_price"] = exec_today.reindex(codes)
rebalance_table = rebalance_table.join(buy_df, how="left").join(sell_df, how="left")
rebalance_table = rebalance_table.join(trade_value_df, how="left").join(trade_cost_df, how="left")
rebalance_table[["buy_shares", "sell_shares", "trade_value", "trade_cost"]] = rebalance_table[
    ["buy_shares", "sell_shares", "trade_value", "trade_cost"]
].fillna(0.0)

rebalance_table["net_trade_shares"] = rebalance_table["buy_shares"] - rebalance_table["sell_shares"]
rebalance_table["net_trade_lots"] = rebalance_table["net_trade_shares"] / LOT_SIZE
rebalance_table["after_shares"] = new_shares.reindex(codes).fillna(0).astype(int)
rebalance_table["after_lots"] = rebalance_table["after_shares"] / LOT_SIZE
rebalance_table["after_market_value"] = after_market_value

def classify_action(x):
    if x > 0:
        return "BUY"
    if x < 0:
        return "SELL"
    return "HOLD"

rebalance_table["suggestion"] = rebalance_table["net_trade_shares"].apply(classify_action)

print("cash before =", round(AVAILABLE_CASH, 2))
print("cash target =", round(target_cash_weight * portfolio_value, 2))
print("cash after  =", round(new_cash, 2))
print("cash after weight =", round(after_cash_weight, 4))
print("trade count =", len(trades_df))

display(rebalance_table.sort_values(["suggestion", "target_weight"], ascending=[True, False]))

In [ ]:
# ========= 仅看需要操作的标的 =========
action_table = rebalance_table[rebalance_table["suggestion"] != "HOLD"].copy()
display(action_table.sort_values("net_trade_shares", ascending=False))

print("交易明细：")
display(trades_df)

In [ ]:
# ========= 图形查看：当前权重 / 目标权重 / 调后权重（含现金） =========
compare_weights = pd.DataFrame({
    "current_weight": current_actual_weights,
    "target_weight": target_weights,
    "after_weight": after_weights,
}).fillna(0.0)

compare_weights.loc["CASH", "current_weight"] = cash_weight
compare_weights.loc["CASH", "target_weight"] = target_cash_weight
compare_weights.loc["CASH", "after_weight"] = after_cash_weight

display(compare_weights.sort_values("target_weight", ascending=False))

compare_weights.plot(kind="bar", figsize=(14, 5))
plt.title("Current vs Target vs After-Rebalance Weights")
plt.ylabel("Weight")
plt.tight_layout()
plt.show()

In [ ]:
# ========= 导出建议 =========
EXPORT_DIR = Path("data/exports_dual_momentum_rebalance")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

current_summary.to_csv(EXPORT_DIR / "dm_rebalance_current_summary.csv")
target_weight_df.to_csv(EXPORT_DIR / "dm_rebalance_target_weights.csv")
rebalance_table.to_csv(EXPORT_DIR / "dm_rebalance_suggestion.csv")
trades_df.to_csv(EXPORT_DIR / "dm_rebalance_trades_detail.csv", index=False)
if len(detail) > 0:
    detail.to_csv(EXPORT_DIR / "dm_rebalance_snapshot_detail.csv")

print("export done")
print("export_dir =", EXPORT_DIR.resolve())

In [ ]:
# ========= 精简版调仓建议（含标的名 + 最新 snapshot 结论） =========
signal_date = pd.Timestamp(latest_signal_date)
signal_date_str = signal_date.strftime("%Y-%m-%d")

# 读取标的名称映射
inst_df = manager.store.get_instruments(listed_only=False)
name_map = dict(zip(inst_df["ts_code"], inst_df["name"]))

# 尝试从交易日历里找下一个交易日
next_trade_date = None
try:
    cal_df = manager.store.get_trade_calendar(
        exchange="SSE",
        start_date=signal_date.strftime("%Y%m%d"),
        end_date="20300101",
        is_open=1,
    )
    future_open_dates = pd.to_datetime(cal_df["cal_date"], format="%Y%m%d", errors="coerce")
    future_open_dates = future_open_dates[future_open_dates > signal_date]
    if len(future_open_dates) > 0:
        next_trade_date = future_open_dates.iloc[0]
except Exception:
    next_trade_date = None

simple_advice = rebalance_table[["current_lots", "after_lots"]].copy()

# 转成展示格式
simple_advice = simple_advice.reset_index().rename(columns={
    "index": "ts_code",
    "current_lots": "调整前手数",
    "after_lots": "调整后手数",
})

# 增加标的名称
simple_advice["name"] = simple_advice["ts_code"].map(name_map)

# 调整列顺序
simple_advice = simple_advice[["ts_code", "name", "调整前手数", "调整后手数"]]

# 手数转整数（若存在小数，先四舍五入再转 int，避免 0.0 / 1.0 这类显示）
simple_advice["调整前手数"] = simple_advice["调整前手数"].round().astype(int)
simple_advice["调整后手数"] = simple_advice["调整后手数"].round().astype(int)

print(f"当前读取到的最新市场数据日期：{signal_date_str}")
print("双动量结论：")
print("  market_regime_on =", market_regime_on)
print("  selected_assets  =", selected_assets)
print("  defensive_used   =", defensive_assets_used)
print("  target_cash_w    =", round(target_cash_weight, 4))

if next_trade_date is not None:
    print(f"建议于下一个交易日（{next_trade_date.strftime('%Y-%m-%d')}）参考如下手数调整：")
else:
    print("建议于下一个交易日参考如下手数调整：")

display(simple_advice)